# Employee Attrition — Sprint 2
## Model Training and Evaluation

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## Load Dataset

In [ ]:
df = pd.read_csv("sprint1_encoded.csv")

In [ ]:
df

In [ ]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Decode Encoded Boolean Columns

In [ ]:
bool_cols = df.select_dtypes(include=[bool]).columns
df[bool_cols] = df[bool_cols].astype(int)
df.dtypes

## Train & Test Data

In [ ]:
x = df.drop("attrition", axis=1)
y = df["attrition"]

In [ ]:
x.shape, y.shape

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

## Feature Scaling

In [ ]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [ ]:
x_train_scaled

In [ ]:
x_test_scaled

## Baseline Model (Logistic Regression)

In [ ]:
baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(x_train_scaled, Y_train)
baseline_pred = baseline_model.predict(x_test_scaled)

In [ ]:
baseline_pred

In [ ]:
baseline_acc = accuracy_score(Y_test, baseline_pred)
print("Baseline Accuracy:", baseline_acc)

In [ ]:
print(classification_report(Y_test, baseline_pred))

In [ ]:
confusion_matrix(Y_test, baseline_pred)

## Training Multiple Models

In [ ]:
models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "KNN"                 : KNeighborsClassifier(),
    "Decision Tree"       : DecisionTreeClassifier(random_state=42),
    "Random Forest"       : RandomForestClassifier(random_state=42),
    "Gradient Boosting"   : GradientBoostingClassifier(random_state=42),
    "SVM"                 : SVC(),
    "Naive Bayes"         : GaussianNB()
}

In [ ]:
models

In [ ]:
scale_required = ["Logistic Regression", "KNN", "SVM", "Naive Bayes"]

In [ ]:
results = []

In [ ]:
for name, model in models.items():

    print("\n" + "="*50)
    print("model :", name)

    if name in scale_required:
        model.fit(x_train_scaled, Y_train)
        train_pred = model.predict(x_train_scaled)
        test_pred = model.predict(x_test_scaled)
    else:
        model.fit(X_train, Y_train)
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

    train_accuracy = accuracy_score(Y_train, train_pred)
    test_accuracy = accuracy_score(Y_test, test_pred)
    precision = precision_score(Y_test, test_pred)
    recall = recall_score(Y_test, test_pred)
    f1 = f1_score(Y_test, test_pred)

    print("Train Accuracy:", round(train_accuracy, 4))
    print("Test Accuracy :", round(test_accuracy, 4))
    print("Precision     :", round(precision, 4))
    print("Recall        :", round(recall, 4))
    print("F1 Score      :", round(f1, 4))

    print("\nConfusion Matrix")
    print(confusion_matrix(Y_test, test_pred))

    results.append([
        name,
        train_accuracy,
        test_accuracy,
        precision,
        recall,
        f1
    ])

In [ ]:
results

## Model Comparison Table

In [ ]:
comparison_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Train Accuracy",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

In [ ]:
comparison_df.head()

In [ ]:
comparison_df.info()

In [ ]:
comparison_df.sort_values(
    by="Test Accuracy",
    ascending=False
)

In [ ]:
comparison_df

## Check Overfitting and Underfitting

- Train Accuracy >> Test Accuracy

In [ ]:
comparison_df["Over Fitting"] = np.where(
    comparison_df["Train Accuracy"] - comparison_df["Test Accuracy"] > 0.05,
    "Yes",
    "No"
)

In [ ]:
comparison_df.head()

In [ ]:
def remarks(row):
    if row["Over Fitting"] == "Yes":
        return "Model Overfits"

    elif row["Test Accuracy"] > 0.9:
        return "Excellent"

    elif row["Test Accuracy"] > 0.8:
        return "Good"

    else:
        return "Needs Improvement"

In [ ]:
comparison_df["Remarks"] = comparison_df.apply(
    remarks,
    axis=1
)

In [ ]:
comparison_df.head()

## Best Performing Model

In [ ]:
best_model = comparison_df.loc[
    comparison_df["Test Accuracy"].idxmax()
]

print(best_model)

## Sprint 2 Conclusion

- Seven classification models were trained and evaluated using Accuracy, Precision, Recall, and F1 Score. Gradient Boosting achieved the highest test accuracy (~75.6%) and F1 Score (~76.9%) with no overfitting. Logistic Regression and SVM both generalised well without overfitting. Decision Tree and Random Forest showed severe overfitting at baseline (100% train accuracy), however Random Forest retained the highest test-set potential and the best ensemble capability. KNN underperformed due to sensitivity to high-dimensional encoded features. Naive Bayes produced the lowest accuracy due to the feature-independence assumption not holding on this dataset.

## Final Choice: Random Forest Classifier